In [5]:
# =====================================================
# HINDI TO HINGLISH TRANSLATION WITH GEMMA 3:4B
# Optimized for RTX 3070 8GB VRAM
# =====================================================

import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from openai import OpenAI
import re
import time
from datetime import datetime
import os
import gc
import torch
import subprocess
import sys
import locale

# =====================================================
# CONFIGURATION
# =====================================================

MODEL = "gemma3:4b"  # or "gemma:2b" if you want smaller
INPUT_FILE = "multilingual_complaints.csv"
OUTPUT_FILE = "complaints_hinglish_final.csv"
CHECKPOINT_FILE = "checkpoint_hinglish.csv"

# OPTIMIZED FOR RTX 3070 8GB
BATCH_SIZE = 8  # Smaller batch size for better quality and memory
MAX_RETRIES = 3
RETRY_DELAY = 2  # seconds

# =====================================================
# OLLAMA MANAGEMENT FUNCTIONS
# =====================================================

def check_ollama_installed():
    """Check if Ollama is installed"""
    try:
        result = subprocess.run(['ollama', '--version'], capture_output=True, text=True, encoding='utf-8', errors='ignore')
        return result.returncode == 0
    except FileNotFoundError:
        return False
    except Exception:
        return False

def pull_ollama_model(model_name):
    """Pull the model using Ollama CLI with proper encoding handling"""
    print(f"\n📥 Pulling model '{model_name}' from Ollama...")
    print("This may take a few minutes depending on your internet speed...")
    print("Progress will be shown below:\n")
    
    try:
        # Use subprocess with proper encoding
        process = subprocess.Popen(
            ['ollama', 'pull', model_name],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            universal_newlines=True,
            bufsize=1,
            encoding='utf-8',
            errors='ignore'  # Ignore characters that can't be decoded
        )
        
        # Print progress in real-time
        for line in process.stdout:
            line = line.strip()
            if line:
                # Clean up the output
                print(f"   {line}")
        
        process.wait()
        
        if process.returncode == 0:
            print(f"\n✅ Successfully pulled '{model_name}'")
            return True
        else:
            print(f"\n❌ Failed to pull '{model_name}'")
            print(f"   Return code: {process.returncode}")
            return False
            
    except UnicodeDecodeError:
        # Fallback: try with different encoding
        try:
            process = subprocess.Popen(
                ['ollama', 'pull', model_name],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                universal_newlines=True,
                bufsize=1,
                encoding='latin-1',
                errors='ignore'
            )
            
            for line in process.stdout:
                line = line.strip()
                if line:
                    print(f"   {line}")
            
            process.wait()
            
            if process.returncode == 0:
                print(f"\n✅ Successfully pulled '{model_name}'")
                return True
            else:
                print(f"\n❌ Failed to pull '{model_name}'")
                return False
        except Exception as e:
            print(f"❌ Error pulling model: {e}")
            return False
            
    except Exception as e:
        print(f"❌ Error pulling model: {e}")
        return False

def list_ollama_models():
    """List available models in Ollama"""
    try:
        result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, encoding='utf-8', errors='ignore')
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            models = []
            for line in lines:
                if line and not line.startswith('NAME'):
                    parts = line.split()
                    if parts:
                        models.append(parts[0])
            return models
        return []
    except Exception:
        return []

def ensure_model_available(model_name):
    """Ensure the model is available, pull if not"""
    # Check if Ollama is installed
    if not check_ollama_installed():
        print("❌ Ollama is not installed!")
        print("Please install Ollama from: https://ollama.ai/download")
        return False
    
    # Check available models
    available_models = list_ollama_models()
    
    # Try to find the model (with or without tag)
    model_found = False
    actual_model = None
    for available in available_models:
        if available.startswith(model_name.split(':')[0]):
            model_found = True
            actual_model = available
            break
    
    if model_found:
        print(f"✅ Model '{actual_model}' is already available")
        return True
    
    # Model not found, try to pull
    print(f"⚠️ Model '{model_name}' not found in Ollama")
    
    # For non-interactive environments, try pulling automatically
    try:
        print("🔄 Attempting to pull model automatically...")
        return pull_ollama_model(model_name)
    except Exception as e:
        print(f"❌ Failed to pull model automatically: {e}")
        print(f"\nPlease pull it manually using:")
        print(f"   ollama pull {model_name}")
        return False

# =====================================================
# OLLAMA CLIENT SETUP
# =====================================================

def create_ollama_client():
    """Create Ollama client with proper error handling"""
    try:
        # Check if Ollama is running
        try:
            import requests
            response = requests.get("http://localhost:11434/api/tags", timeout=5)
            if response.status_code != 200:
                print("❌ Ollama is not running properly")
                return None, False
        except:
            print("❌ Cannot connect to Ollama. Is it running?")
            print("   Start Ollama with: ollama serve")
            return None, False
        
        client = OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )
        
        # Test connection with a simple request
        test_response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": "Hello"}],
            max_tokens=5,
            timeout=10
        )
        
        return client, True
    except Exception as e:
        print(f"❌ Ollama connection error: {e}")
        print("\nTry restarting Ollama:")
        print("   ollama serve")
        return None, False

# =====================================================
# ENHANCED SYSTEM PROMPT WITH EXAMPLES
# =====================================================

SYSTEM_PROMPT = """You are a Hindi-to-Hinglish translator. Convert Hindi text to natural Hinglish (Hindi written in Roman script with English mixed in).

CRITICAL RULES:
1. Use ONLY Roman script (English alphabet)
2. Write in natural WhatsApp/chat style
3. Keep common English words as-is (like "phone", "internet", "bill", "customer")
4. Keep the exact meaning and emotion
5. DO NOT translate to pure English - it should be Hinglish
6. Return ONLY a JSON array of strings
7. Each string should be one complaint exactly as provided

EXAMPLES OF CORRECT HINGLISH:

Hindi: "मेरे घर के सामने कचरा पड़ा है।"
Hinglish: "Mere ghar ke saamne kachra pada hai."

Hindi: "मुझे पिछले 3 महीने से बिजली का बिल नहीं आया है।"
Hinglish: "Mujhe pichle 3 mahine se bijli ka bill nahi aaya hai."

Hindi: "सड़क पर गड्ढे हैं जिससे राहगीरों को परेशानी होती है।"
Hinglish: "Sadak par gadde hain jisse raahgeeron ko pareshani hoti hai."

Hindi: "आपके कस्टमर केयर नंबर पर कॉल नहीं लग रहा है।"
Hinglish: "Aapke customer care number par call nahi lag raha hai."

Hindi: "मेरा इंटरनेट कनेक्शन बहुत धीमा है और बार-बार डिस्कनेक्ट होता है।"
Hinglish: "Mera internet connection bahut slow hai aur baar-baar disconnect hota hai."

Remember: The output MUST be a JSON array. For multiple complaints, the array should have the same number of elements as input complaints.
Example output format: ["Hinglish complaint 1", "Hinglish complaint 2", "Hinglish complaint 3"]"""

# =====================================================
# USER PROMPT TEMPLATE
# =====================================================

def create_user_prompt(complaints):
    """Create user prompt with numbered complaints"""
    prompt = f"Convert these {len(complaints)} Hindi complaints to natural Hinglish. Return ONLY a JSON array with exactly {len(complaints)} items.\n\n"
    
    for i, text in enumerate(complaints, 1):
        if text and text.strip():  # Only add non-empty complaints
            prompt += f"{i}. {text.strip()}\n"
        else:
            prompt += f"{i}. [empty complaint]\n"
    
    return prompt

# =====================================================
# VALIDATION FUNCTIONS
# =====================================================

def clean_response(text):
    """Clean and extract JSON from response"""
    if not text:
        return ""
    
    # Remove markdown
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = re.sub(r'^json\s*', '', text)
    
    # Find JSON array
    start = text.find('[')
    end = text.rfind(']')
    
    if start != -1 and end != -1 and start < end:
        return text[start:end+1]
    
    return text

def fix_malformed_json(text):
    """Attempt to fix common JSON issues"""
    try:
        # Try to extract strings
        pattern = r'"([^"]*)"'
        matches = re.findall(pattern, text)
        if matches:
            return json.dumps(matches)
    except:
        pass
    
    # Try to split by newlines and clean
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    cleaned_lines = []
    
    for line in lines:
        # Remove numbering
        line = re.sub(r'^\d+\.\s*', '', line)
        # Remove quotes if present
        line = line.strip('"')
        if line:
            cleaned_lines.append(line)
    
    if cleaned_lines:
        return json.dumps(cleaned_lines)
    
    return None

# =====================================================
# TRANSLATION FUNCTION
# =====================================================

def translate_batch(client, batch_complaints, batch_idx):
    """Translate a batch of complaints with retries"""
    
    if not batch_complaints or all(not str(c).strip() for c in batch_complaints):
        return ["Empty complaint"] * len(batch_complaints)
    
    # Remove empty complaints and track indices
    non_empty_indices = [i for i, c in enumerate(batch_complaints) if str(c).strip()]
    
    if not non_empty_indices:
        return ["Empty complaint"] * len(batch_complaints)
    
    # Create prompt with non-empty complaints
    non_empty_complaints = [str(batch_complaints[i]) for i in non_empty_indices]
    prompt = create_user_prompt(non_empty_complaints)
    
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=0.1,  # Low temperature for consistency
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ],
                extra_body={
                    "keep_alive": "5m",
                    "num_predict": 4096,
                    "num_ctx": 2048  # Context window for batch
                },
                timeout=60  # Timeout after 60 seconds
            )
            
            result = response.choices[0].message.content.strip()
            
            if not result:
                raise ValueError("Empty response from model")
            
            # Clean and extract JSON
            cleaned = clean_response(result)
            
            # Try to parse JSON
            try:
                translations = json.loads(cleaned)
            except json.JSONDecodeError:
                # Try to fix malformed JSON
                fixed = fix_malformed_json(cleaned)
                if fixed:
                    translations = json.loads(fixed)
                else:
                    # Try to parse by splitting
                    parts = re.findall(r'"([^"]*)"', cleaned)
                    if parts:
                        translations = parts
                    else:
                        raise ValueError("Could not parse JSON")
            
            # Handle single string response
            if isinstance(translations, str):
                translations = [translations]
            
            # Ensure we have correct number of translations
            if len(translations) != len(non_empty_complaints):
                # Try to match count
                if len(translations) > len(non_empty_complaints):
                    translations = translations[:len(non_empty_complaints)]
                else:
                    # Pad with empty strings
                    translations.extend([""] * (len(non_empty_complaints) - len(translations)))
            
            # Create full output with empty complaints in correct positions
            full_output = [""] * len(batch_complaints)
            for idx, orig_idx in enumerate(non_empty_indices):
                if idx < len(translations):
                    full_output[orig_idx] = str(translations[idx]).strip()
                else:
                    full_output[orig_idx] = ""
            
            return full_output
            
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = RETRY_DELAY * (attempt + 1)
                print(f"\n   ⏳ Retry {attempt + 1}/{MAX_RETRIES} for batch {batch_idx} in {wait_time}s...")
                time.sleep(wait_time)
                continue
            else:
                print(f"\n❌ Batch {batch_idx} failed after {MAX_RETRIES} attempts: {e}")
                # Return empty translations for failed batch
                return [""] * len(batch_complaints)
    
    return [""] * len(batch_complaints)

# =====================================================
# MAIN PROCESSING FUNCTION
# =====================================================

def main():
    print("=" * 60)
    print("HINDI TO HINGLISH TRANSLATION")
    print("=" * 60)
    print(f"Model: {MODEL}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Device: GPU (RTX 3070 8GB)")
    print("=" * 60)
    
    # Check and ensure model is available
    print("\n📦 Checking model availability...")
    if not ensure_model_available(MODEL):
        print("\n❌ Cannot proceed without model. Please install Ollama and pull the model.")
        print(f"   Manual command: ollama pull {MODEL}")
        return
    
    # Create client
    print("\n🔗 Connecting to Ollama...")
    client, success = create_ollama_client()
    if not success:
        print("\n❌ Cannot connect to Ollama. Please make sure Ollama is running.")
        print("   Start Ollama with: ollama serve")
        return
    
    # Load data
    print("\n📂 Loading data...")
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: {INPUT_FILE} not found!")
        return
    
    try:
        df = pd.read_csv(INPUT_FILE)
        total_rows = len(df)
        print(f"✅ Loaded {total_rows} complaints")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    # Check if we have a checkpoint
    start_idx = 0
    all_hinglish = []
    
    if os.path.exists(CHECKPOINT_FILE):
        try:
            checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
            if 'complaint_hinglish' in checkpoint_df.columns:
                start_idx = len(checkpoint_df)
                all_hinglish = checkpoint_df['complaint_hinglish'].tolist()
                print(f"🔄 Resuming from checkpoint: {start_idx} rows already processed")
                
                # Verify we have matching number of rows
                if len(all_hinglish) != start_idx:
                    print(f"⚠️ Checkpoint mismatch. Starting fresh...")
                    start_idx = 0
                    all_hinglish = []
        except Exception as e:
            print(f"⚠️ Could not read checkpoint: {e}")
            start_idx = 0
            all_hinglish = []
    
    # Prepare texts
    texts = df['Text_hi'].fillna('').astype(str).tolist()
    
    # Process batches
    remaining = total_rows - start_idx
    if remaining <= 0:
        print("✅ All complaints already processed!")
        return
    
    print(f"\n🔄 Processing {remaining} remaining complaints...")
    total_batches = (remaining + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"Total batches: {total_batches}")
    print("-" * 60)
    
    # Create progress bar
    pbar = tqdm(
        total=remaining,
        desc="Translating",
        unit="complaints",
        ncols=100
    )
    
    # Process batches
    for batch_start in range(start_idx, total_rows, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, total_rows)
        batch_complaints = texts[batch_start:batch_end]
        batch_idx = batch_start // BATCH_SIZE
        
        # Translate batch
        start_time = time.time()
        translations = translate_batch(client, batch_complaints, batch_idx)
        elapsed_time = time.time() - start_time
        
        # Add to results
        all_hinglish.extend(translations)
        
        # Update progress
        pbar.update(len(translations))
        
        # Show current progress occasionally
        current_total = len(all_hinglish)
        if current_total % 50 == 0 or current_total == total_rows:
            print(f"\n📊 Progress: {current_total}/{total_rows} ({current_total/total_rows*100:.1f}%)")
            print(f"⏱️ Last batch: {elapsed_time:.1f}s")
            
            # Show sample of current batch
            if translations:
                sample_idx = min(2, len(translations) - 1)
                for i in range(sample_idx):
                    if translations[i] and translations[i].strip():
                        print(f"   Sample {i+1}: {translations[i][:60]}...")
        
        # Save checkpoint every 5 batches or after significant progress
        if (batch_idx % 5 == 0 or batch_end >= total_rows) and len(all_hinglish) > 0:
            try:
                temp_df = df.iloc[:len(all_hinglish)].copy()
                temp_df['complaint_hinglish'] = all_hinglish
                temp_df.to_csv(CHECKPOINT_FILE, index=False)
            except Exception as e:
                print(f"⚠️ Could not save checkpoint: {e}")
        
        # Clear memory periodically
        if batch_idx % 10 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    pbar.close()
    
    # Save final output
    print("\n💾 Saving final output...")
    try:
        df['complaint_hinglish'] = all_hinglish
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"✅ Translation completed!")
        print(f"📁 Output saved to: {OUTPUT_FILE}")
        
        # Show statistics
        successful = sum(1 for x in all_hinglish if x and str(x).strip())
        empty = total_rows - successful
        print(f"\n📊 Statistics:")
        print(f"   Total complaints: {total_rows}")
        print(f"   Successfully translated: {successful} ({successful/total_rows*100:.1f}%)")
        print(f"   Empty/failed: {empty} ({empty/total_rows*100:.1f}%)")
        
        # Show sample translations
        print(f"\n📝 Sample Translations:")
        sample_count = min(5, total_rows)
        for i in range(sample_count):
            if all_hinglish[i] and str(all_hinglish[i]).strip():
                print(f"\n{i+1}. Hindi: {str(df['Text_hi'].iloc[i])[:80]}...")
                print(f"   Hinglish: {str(all_hinglish[i])[:80]}...")
        
    except Exception as e:
        print(f"❌ Error saving output: {e}")
    
    print("\n" + "=" * 60)
    print("🎉 PROCESS COMPLETE!")
    print("=" * 60)

# =====================================================
# CHECK GPU AVAILABILITY AND OLLAMA
# =====================================================

def check_environment():
    """Check if environment is properly set up"""
    print("\n🔍 Environment Check:")
    
    # Check PyTorch and CUDA
    try:
        import torch
        if torch.cuda.is_available():
            print(f"✅ PyTorch CUDA available: {torch.cuda.get_device_name(0)}")
            print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        else:
            print("⚠️ PyTorch CUDA not available")
    except:
        print("⚠️ PyTorch not installed")
    
    # Check Ollama installation
    try:
        result = subprocess.run(['ollama', '--version'], capture_output=True, text=True, encoding='utf-8', errors='ignore')
        if result.returncode == 0:
            print(f"✅ Ollama is installed")
            print(f"   Version: {result.stdout.strip()}")
            
            # Check if model exists
            try:
                model_check = subprocess.run(['ollama', 'list'], capture_output=True, text=True, encoding='utf-8', errors='ignore')
                if model_check.returncode == 0:
                    lines = model_check.stdout.strip().split('\n')
                    models = []
                    for line in lines:
                        if line and not line.startswith('NAME'):
                            parts = line.split()
                            if parts:
                                models.append(parts[0])
                    
                    if any(MODEL.split(':')[0] in m for m in models):
                        print(f"✅ Model '{MODEL}' is available")
                    else:
                        print(f"⚠️ Model '{MODEL}' not found. Will attempt to pull.")
                else:
                    print("⚠️ Could not list Ollama models")
            except Exception as e:
                print(f"⚠️ Could not check models: {e}")
        else:
            print("❌ Ollama not installed")
            return False
    except Exception as e:
        print(f"❌ Ollama error: {e}")
        return False
    
    # Check input file
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Input file not found: {INPUT_FILE}")
        return False
    
    print(f"✅ Input file found: {INPUT_FILE}")
    return True

# =====================================================
# OPTIONAL: QUICK TEST WITH SAMPLE
# =====================================================

def quick_test():
    """Quick test with a few sample complaints"""
    print("\n" + "=" * 60)
    print("QUICK TEST WITH SAMPLE DATA")
    print("=" * 60)
    
    # Ensure model is available
    if not ensure_model_available(MODEL):
        return
    
    # Create client
    client, success = create_ollama_client()
    if not success:
        print("❌ Cannot connect to Ollama")
        return
    
    test_complaints = [
        "मेरे घर के सामने कचरा पड़ा है।",
        "मुझे पिछले 3 महीने से बिजली का बिल नहीं आया है।",
        "सड़क पर गड्ढे हैं जिससे राहगीरों को परेशानी होती है।",
        "आपके कस्टमर केयर नंबर पर कॉल नहीं लग रहा है।"
    ]
    
    print("\n📝 Test Complaints:")
    for i, complaint in enumerate(test_complaints, 1):
        print(f"{i}. {complaint}")
    
    print("\n🔄 Translating test batch...")
    translated = translate_batch(client, test_complaints, 0)
    
    print("\n✅ Translation Results:")
    for i, (original, translated_text) in enumerate(zip(test_complaints, translated), 1):
        print(f"{i}. Original: {original}")
        print(f"   Translated: {translated_text}")
        print()

# =====================================================
# RUN MAIN
# =====================================================

if __name__ == "__main__":
    try:
        # First check environment
        if check_environment():
            print("\n✅ Environment check passed!")
            
            # Ask if user wants to run quick test
            try:
                response = input("\nDo you want to run a quick test first? (y/n): ")
                if response.lower() == 'y':
                    quick_test()
            except:
                print("\n⚠️ Could not get user input. Running full process...")
            
            # Ask if user wants to proceed with full translation
            try:
                response = input("\nProceed with full translation? (y/n): ")
                if response.lower() == 'y':
                    main()
                else:
                    print("✅ Exiting. You can run main() later.")
            except:
                print("\n⚠️ Could not get user input. Proceeding with full translation...")
                main()
        else:
            print("\n❌ Environment check failed. Please fix issues before running.")
            print("\n📝 Troubleshooting steps:")
            print("1. Install Ollama: https://ollama.ai/download")
            print("2. Pull the model manually: ollama pull gemma3:4b")
            print("3. Start Ollama: ollama serve")
            print("4. Verify Ollama is running: curl http://localhost:11434/api/tags")
            
    except KeyboardInterrupt:
        print("\n\n⏹️ Process interrupted by user")
        print("💾 Checkpoint saved - you can resume later")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        import traceback
        traceback.print_exc()


🔍 Environment Check:
✅ PyTorch CUDA available: NVIDIA GeForce RTX 3070
   GPU Memory: 8.6 GB
✅ Ollama is installed
   Version: ollama version is 0.32.9
✅ Model 'gemma3:4b' is available
✅ Input file found: multilingual_complaints.csv

✅ Environment check passed!



Do you want to run a quick test first? (y/n):  y



QUICK TEST WITH SAMPLE DATA
✅ Model 'gemma3:4b' is already available

📝 Test Complaints:
1. मेरे घर के सामने कचरा पड़ा है।
2. मुझे पिछले 3 महीने से बिजली का बिल नहीं आया है।
3. सड़क पर गड्ढे हैं जिससे राहगीरों को परेशानी होती है।
4. आपके कस्टमर केयर नंबर पर कॉल नहीं लग रहा है।

🔄 Translating test batch...

✅ Translation Results:
1. Original: मेरे घर के सामने कचरा पड़ा है।
   Translated: Mere ghar ke saamne kachra pada hai.

2. Original: मुझे पिछले 3 महीने से बिजली का बिल नहीं आया है।
   Translated: Mujhe pichle 3 mahine se bijli ka bill nahi aaya hai.

3. Original: सड़क पर गड्ढे हैं जिससे राहगीरों को परेशानी होती है।
   Translated: Sadak par gadde hain jisse raahgeeron ko pareshani hoti hai.

4. Original: आपके कस्टमर केयर नंबर पर कॉल नहीं लग रहा है।
   Translated: Aapke customer care number par call nahi lag raha hai.




Proceed with full translation? (y/n):  y


HINDI TO HINGLISH TRANSLATION
Model: gemma3:4b
Batch Size: 8
Device: GPU (RTX 3070 8GB)

📦 Checking model availability...
✅ Model 'gemma3:4b' is already available

🔗 Connecting to Ollama...

📂 Loading data...
✅ Loaded 15000 complaints

🔄 Processing 15000 remaining complaints...
Total batches: 1875
------------------------------------------------------------


Translating:   1%|▍                                   | 200/15000 [02:23<2:42:19,  1.52complaints/s]

... (147 outputs left out)

Translating: 100%|██████████████████████████████████| 15000/15000 [2:54:23<00:00,  1.14s/complaints]


📊 Progress: 15000/15000 (100.0%)
⏱️ Last batch: 8.7s
   Sample 1: Kota mein coaching kendron ke ek group ka pass footpath par ...
   Sample 2: Baraikpur mein ek Pashu Shala ke bahar ki road gobhar, ghaas...


Translating: 100%|██████████████████████████████████| 15000/15000 [2:54:23<00:00,  1.43complaints/s]



💾 Saving final output...
✅ Translation completed!
📁 Output saved to: complaints_hinglish_final.csv

📊 Statistics:
   Total complaints: 15000
   Successfully translated: 14984 (99.9%)
   Empty/failed: 16 (0.1%)

📝 Sample Translations:

1. Hindi: हाल की बारिश के बाद हमारी सड़क के पास बिजली वितरण पोल थोड़ा झुका हुआ है। इससे को...
   Hinglish: Hall ki baarish ke baad humari sadak ke paas bijli distribution pole thoda jhuka...

2. Hindi: बिजली के खंभे से जुड़ी पहचान प्लेट गिर गई है, जिससे दोषों की सटीक रिपोर्ट करना म...
   Hinglish: Bijli ke khambe se judi ideni plate gir gayi hai, jisse doshon ki tarki report k...

3. Hindi: सड़क के बगल में एक विद्युत जंक्शन बॉक्स का आवरण आंशिक रूप से खुला होता है। हालां...
   Hinglish: Sadak ke bagle mein ek vishvasti junction box ka avataran partiale khula hota ha...

4. Hindi: आवासीय लेन के पास बिजली के तारों को ऊपर रखने वाला एक सहायक क्लैंप ढीला दिखाई देत...
   Hinglish: Awasiyab lane ke paas bijli ke tairon ko upar rakhna wala ek sahayak clamp dhila.

In [4]:
# ============================================================================
# OLLAMA CONNECTION FIX - COMPLETE IMPLEMENTATION
# ============================================================================

import subprocess
import time
import requests
import socket
import os
import sys

class OllamaManager:
    """
    Comprehensive Ollama connection manager with auto-recovery
    """
    
    def __init__(self, host='localhost', port=11434, timeout=30):
        """
        Initialize Ollama manager
        
        Args:
            host: Ollama host
            port: Ollama port
            timeout: Connection timeout in seconds
        """
        self.host = host
        self.port = port
        self.timeout = timeout
        self.base_url = f"http://{host}:{port}"
        self.is_connected = False
        
    def check_connection(self, max_retries=3, delay=2):
        """
        Check if Ollama is running and responsive
        
        Args:
            max_retries: Maximum number of connection attempts
            delay: Delay between attempts in seconds
            
        Returns:
            bool: True if connected, False otherwise
        """
        for attempt in range(max_retries):
            try:
                # Check if port is open
                sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                sock.settimeout(2)
                result = sock.connect_ex((self.host, self.port))
                sock.close()
                
                if result == 0:
                    # Port is open, test API
                    response = requests.get(
                        f"{self.base_url}/api/tags",
                        timeout=5
                    )
                    if response.status_code == 200:
                        print(f"✅ Ollama is running and responsive")
                        self.is_connected = True
                        return True
                else:
                    print(f"⚠️  Attempt {attempt + 1}/{max_retries}: Port {self.port} is not open")
                    
            except requests.exceptions.Timeout:
                print(f"⚠️  Attempt {attempt + 1}/{max_retries}: Connection timeout")
            except requests.exceptions.ConnectionError:
                print(f"⚠️  Attempt {attempt + 1}/{max_retries}: Connection refused")
            except Exception as e:
                print(f"⚠️  Attempt {attempt + 1}/{max_retries}: {str(e)}")
            
            if attempt < max_retries - 1:
                print(f"⏳ Waiting {delay} seconds before retry...")
                time.sleep(delay)
        
        self.is_connected = False
        return False
    
    def start_ollama(self):
        """
        Attempt to start Ollama service
        
        Returns:
            bool: True if started successfully, False otherwise
        """
        print("\n🔄 Attempting to start Ollama...")
        
        # Check if Ollama is installed
        try:
            subprocess.run(['ollama', '--version'], 
                          capture_output=True, 
                          check=True)
            print("✅ Ollama is installed")
        except (subprocess.CalledProcessError, FileNotFoundError):
            print("❌ Ollama is not installed. Please install Ollama first.")
            print("   Visit: https://ollama.ai/")
            return False
        
        # Try different methods to start Ollama
        methods = [
            # Method 1: Start in background
            lambda: subprocess.Popen(
                ['ollama', 'serve'],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                start_new_session=True
            ),
            # Method 2: Using systemd (Linux)
            lambda: subprocess.run(
                ['sudo', 'systemctl', 'start', 'ollama'],
                capture_output=True
            ),
            # Method 3: Using launchctl (macOS)
            lambda: subprocess.run(
                ['launchctl', 'start', 'ollama'],
                capture_output=True
            )
        ]
        
        for method in methods:
            try:
                method()
                print("✅ Ollama start command executed")
                break
            except Exception as e:
                print(f"⚠️  Method failed: {str(e)}")
                continue
        
        # Wait for service to start
        print("⏳ Waiting for Ollama to start...")
        for i in range(10):
            if self.check_connection(max_retries=1):
                return True
            time.sleep(2)
            print(f"   Waiting... {i+1}/10")
        
        return False
    
    def ensure_connection(self):
        """
        Ensure Ollama is running and connected
        
        Returns:
            bool: True if connected, False otherwise
        """
        print("\n🔗 Checking Ollama connection...")
        
        # First check connection
        if self.check_connection():
            return True
        
        # Connection failed, try to start Ollama
        print("\n⚠️  Ollama is not running. Attempting to start...")
        
        if self.start_ollama():
            print("✅ Ollama started successfully!")
            return True
        
        print("\n❌ Could not connect to Ollama. Please manually start it with:")
        print("   ollama serve")
        print("\nThen run this script again.")
        
        return False
    
    def pull_model(self, model_name='gemma3:4b'):
        """
        Pull a model from Ollama registry
        
        Args:
            model_name: Name of the model to pull
            
        Returns:
            bool: True if pulled successfully, False otherwise
        """
        print(f"\n📦 Pulling model '{model_name}'...")
        
        try:
            # Check if model already exists
            response = requests.get(f"{self.base_url}/api/tags", timeout=10)
            if response.status_code == 200:
                models = response.json().get('models', [])
                for model in models:
                    if model.get('name', '') == model_name:
                        print(f"✅ Model '{model_name}' already exists")
                        return True
            
            # Pull the model
            print(f"⏳ Pulling model '{model_name}'... This may take a while.")
            
            # Use subprocess for better progress visibility
            process = subprocess.Popen(
                ['ollama', 'pull', model_name],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                bufsize=1
            )
            
            # Print progress
            for line in process.stdout:
                print(line.strip())
            
            process.wait()
            
            if process.returncode == 0:
                print(f"✅ Model '{model_name}' pulled successfully!")
                return True
            else:
                print(f"❌ Failed to pull model '{model_name}'")
                return False
                
        except Exception as e:
            print(f"❌ Error pulling model: {str(e)}")
            return False
    
    def get_model_list(self):
        """
        Get list of available models
        
        Returns:
            list: List of available model names
        """
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=10)
            if response.status_code == 200:
                models = response.json().get('models', [])
                return [model.get('name', '') for model in models]
            return []
        except:
            return []
    
    def generate_with_ollama(self, prompt, model_name='gemma3:4b', 
                            temperature=0.7, max_tokens=512):
        """
        Generate text using Ollama
        
        Args:
            prompt: Input prompt
            model_name: Model to use
            temperature: Temperature for generation
            max_tokens: Maximum tokens to generate
            
        Returns:
            str: Generated text or None on failure
        """
        if not self.is_connected:
            print("❌ Not connected to Ollama")
            return None
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": model_name,
                    "prompt": prompt,
                    "stream": False,
                    "temperature": temperature,
                    "max_tokens": max_tokens
                },
                timeout=self.timeout
            )
            
            if response.status_code == 200:
                result = response.json()
                return result.get('response', '')
            else:
                print(f"❌ API error: {response.status_code}")
                return None
                
        except requests.exceptions.Timeout:
            print(f"❌ Request timed out after {self.timeout} seconds")
            return None
        except Exception as e:
            print(f"❌ Error: {str(e)}")
            return None


# ============================================================================
# HINDI TO HINGLISH TRANSLATOR WITH OLLAMA FIX
# ============================================================================

class HindiToHinglishTranslator:
    """
    Translator using Ollama with automatic connection management
    """
    
    def __init__(self, model_name='gemma3:4b', batch_size=8):
        """
        Initialize translator
        
        Args:
            model_name: Ollama model to use
            batch_size: Batch size for translation
        """
        self.model_name = model_name
        self.batch_size = batch_size
        self.ollama = OllamaManager(timeout=60)  # Longer timeout for generation
        
    def translate(self, hindi_texts, progress_callback=None):
        """
        Translate Hindi text to Hinglish
        
        Args:
            hindi_texts: List of Hindi texts
            progress_callback: Optional callback for progress updates
            
        Returns:
            list: Translated texts
        """
        if not isinstance(hindi_texts, list):
            hindi_texts = [hindi_texts]
        
        if not self.ollama.ensure_connection():
            print("\n❌ Cannot proceed with translation. Please fix Ollama connection.")
            return None
        
        # Ensure model is available
        if not self.ollama.pull_model(self.model_name):
            print(f"\n❌ Cannot proceed. Model '{self.model_name}' not available.")
            return None
        
        translated = []
        total = len(hindi_texts)
        
        print(f"\n🔀 Translating {total} texts...")
        
        for i in range(0, total, self.batch_size):
            batch = hindi_texts[i:i+self.batch_size]
            batch_translated = []
            
            for j, text in enumerate(batch):
                if not text or len(text.strip()) == 0:
                    batch_translated.append("")
                    continue
                
                # Create prompt
                prompt = f"""You are a translator converting Hindi text to Hinglish (Hindi written in Roman script with English mix).

Input Hindi text: {text}

Rules:
1. Write in Roman script (English alphabet)
2. Keep commonly used Hindi words (like 'hai', 'hum', 'aap', 'main', 'tum')
3. Use English words for technical terms
4. Maintain natural conversational flow
5. Do NOT use Devanagari script

Hinglish translation:"""
                
                # Get translation
                response = self.ollama.generate_with_ollama(
                    prompt=prompt,
                    model_name=self.model_name,
                    temperature=0.3,  # Lower temperature for consistency
                    max_tokens=256
                )
                
                if response:
                    # Clean response (remove extra text if model added)
                    response = response.strip()
                    if 'Hinglish translation:' in response:
                        response = response.split('Hinglish translation:')[-1].strip()
                    batch_translated.append(response)
                else:
                    batch_translated.append(text)  # Return original on failure
                    print(f"⚠️  Translation failed for: {text[:50]}...")
                
                # Update progress
                if progress_callback:
                    progress_callback(i + j + 1, total)
            
            translated.extend(batch_translated)
            
            # Small delay to avoid rate limiting
            time.sleep(0.1)
        
        return translated


# ============================================================================
# MAIN EXECUTION WITH FIXES
# ============================================================================

def main():
    """
    Main execution with Ollama connection handling
    """
    print("="*80)
    print("HINDI TO HINGLISH TRANSLATOR")
    print("="*80)
    
    # Check Ollama availability first
    ollama = OllamaManager(timeout=30)
    
    if not ollama.ensure_connection():
        print("\n❌ Cannot connect to Ollama.")
        print("\nPlease manually start Ollama with:")
        print("   ollama serve")
        print("\nIn a new terminal window, then run this script again.")
        sys.exit(1)
    
    # Check if model exists, pull if needed
    if not ollama.pull_model('gemma3:4b'):
        print("\n❌ Failed to pull model.")
        print("Please pull the model manually with:")
        print("   ollama pull gemma3:4b")
        sys.exit(1)
    
    # Sample Hindi texts
    sample_texts = [
        "नमस्ते, आप कैसे हैं?",
        "मैं आज बहुत खुश हूँ क्योंकि मेरा काम पूरा हो गया।",
        "क्या आप मुझे बता सकते हैं कि यह जगह कहाँ है?",
        "हमें कल सुबह 8 बजे मिलना है।",
        "यह खाना बहुत स्वादिष्ट है, धन्यवाद!"
    ]
    
    print("\n📝 Sample texts to translate:")
    for i, text in enumerate(sample_texts):
        print(f"  {i+1}. {text}")
    
    # Initialize translator
    translator = HindiToHinglishTranslator(
        model_name='gemma3:4b',
        batch_size=4
    )
    
    # Progress callback
    def update_progress(current, total):
        percentage = int((current / total) * 100)
        bar_length = 30
        filled = int(bar_length * current // total)
        bar = '█' * filled + '░' * (bar_length - filled)
        print(f"\rProgress: [{bar}] {percentage}% ({current}/{total})", end='')
        if current == total:
            print()  # New line when complete
    
    # Translate
    print("\n🔄 Starting translation...")
    translated = translator.translate(sample_texts, progress_callback=update_progress)
    
    if translated:
        print("\n" + "="*80)
        print("TRANSLATION RESULTS")
        print("="*80)
        
        for i, (original, translation) in enumerate(zip(sample_texts, translated)):
            print(f"\n{i+1}. Original: {original}")
            print(f"   Translated: {translation}")
    
    print("\n" + "="*80)
    print("TRANSLATION COMPLETE")
    print("="*80)


# ============================================================================
# ALTERNATIVE: USING REQUESTS DIRECTLY (Simpler Fix)
# ============================================================================

def check_ollama_simple():
    """
    Simple Ollama connection check and fix
    """
    try:
        # Try to connect
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            print("✅ Ollama is running!")
            return True
        else:
            print(f"❌ Ollama returned status: {response.status_code}")
            return False
    except requests.exceptions.ConnectionError:
        print("❌ Ollama is not running")
        print("\n💡 To fix this:")
        print("1. Open a new terminal/command prompt")
        print("2. Run: ollama serve")
        print("3. Wait for Ollama to start")
        print("4. Run this script again")
        
        # Try to start automatically
        start_choice = input("\nTry to start Ollama automatically? (y/n): ")
        if start_choice.lower() == 'y':
            subprocess.Popen(['ollama', 'serve'], 
                           stdout=subprocess.DEVNULL, 
                           stderr=subprocess.DEVNULL,
                           start_new_session=True)
            print("⏳ Waiting for Ollama to start...")
            time.sleep(5)
            
            # Check again
            try:
                response = requests.get("http://localhost:11434/api/tags", timeout=5)
                if response.status_code == 200:
                    print("✅ Ollama started successfully!")
                    return True
            except:
                pass
            print("❌ Could not start Ollama automatically")
        return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False


# ============================================================================
# RUN THE TRANSLATOR
# ============================================================================

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n🛑 Translation interrupted by user.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

HINDI TO HINGLISH TRANSLATOR

🔗 Checking Ollama connection...
✅ Ollama is running and responsive

📦 Pulling model 'gemma3:4b'...
✅ Model 'gemma3:4b' already exists

📝 Sample texts to translate:
  1. नमस्ते, आप कैसे हैं?
  2. मैं आज बहुत खुश हूँ क्योंकि मेरा काम पूरा हो गया।
  3. क्या आप मुझे बता सकते हैं कि यह जगह कहाँ है?
  4. हमें कल सुबह 8 बजे मिलना है।
  5. यह खाना बहुत स्वादिष्ट है, धन्यवाद!

🔄 Starting translation...

🔗 Checking Ollama connection...
✅ Ollama is running and responsive

📦 Pulling model 'gemma3:4b'...
✅ Model 'gemma3:4b' already exists

🔀 Translating 5 texts...
Progress: [██████████████████████████████] 100% (5/5)

TRANSLATION RESULTS

1. Original: नमस्ते, आप कैसे हैं?
   Translated: Namaste, aap kaise hain? How are you?

2. Original: मैं आज बहुत खुश हूँ क्योंकि मेरा काम पूरा हो गया।
   Translated: Main aaj bahut khush hu, kyunki mera kaam poora ho gaya.

3. Original: क्या आप मुझे बता सकते हैं कि यह जगह कहाँ है?
   Translated: Kya aap mujhe bata sakte hain ki yeh ja